# AQD operational pipeline

## Explanation and usage

Use this notebook to run the AQD workflow end-to-end in small, explicit steps.
You need a valid `inst_deploy_id`; if you do not know it, use `inst_deploy_id_finder.ipynb` first.

Suggested run order for operators:
1. Run Imports.
2. Run Setup.
3. Run ID lookup to validate inputs and confirm metadata for the selected deployment.
4. If you need to discover an ID first, run `inst_deploy_id_finder.ipynb` before this notebook.
5. Continue to processing stages (`proc_1`, `proc_2`, `imos_delivery`).

Cells are intentionally separated so you can rerun only the step you are working on.

In [3]:
import os
import sys
from pathlib import Path
import importlib
import subprocess

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

TOOLS_DIR = Path.cwd().resolve().parent
if not (TOOLS_DIR / "tools").exists():
    candidate = Path.cwd().resolve()
    if (candidate / "tools").exists():
        TOOLS_DIR = candidate
    elif (candidate / "mooring_proc" / "tools").exists():
        TOOLS_DIR = candidate / "mooring_proc"

if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from tools.helpers import plot_data_by_qc
from tools.workflows.run_imos_delivery import run_imos_delivery
from tools.workflows.run_proc1 import run_proc1
from tools.workflows.run_proc2 import run_proc2
import tools.database_lookup as database_lookup
import pandas as pd
from tools.parsers.read_aqd import read_aqd


## Setup

Set the deployment identifier and optional source override before running the workflow cells you need.

This notebook follows the same pattern as the SBE37 workflow:
- `proc_1` creates the intermediate IMOS FV00 product
- `proc_2` creates the final IMOS FV01 product candidate
- `imos_delivery` validates and stages only the FV01 result


### Working directory

Edit this path if you want the notebook to resolve relative data paths from a different root, not the repository.


In [ ]:
# Working directory
working_directory = "/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data"
os.chdir(working_directory)
print(f"Working directory: {os.getcwd()}")


### Set permissions
This step sets the umask, with options to add at different steps if required.

NOTE: umask sets the process umask to the value you pass and returns the previous umask value

In [ ]:
os.umask(0o002)

# umask 0o002 gives files 664 (-rw-rw-r--)
# umask 0o022 gives files 644 (-rw-r--r--)


### Locate instrument - files and metadata
Set the deployment identifier and instrument or use optional source override (source path+file)

In [ ]:
# Required AQD deployment identifier from the metadata table.
inst_deploy_id = ""

# Keep this notebook scoped to AQD instruments.
instrument = "AQD"

# Optional override for the raw input source file or directory.
proc1_source_path = None


This cell turns the options above into the shared workflow configuration used by the processing stages.


In [ ]:
def build_workflow_config():
    import tools.database_lookup as database_lookup
    database_lookup = importlib.reload(database_lookup)
    metadata_csv = str(database_lookup.DEFAULT_METADATA_CSV_PATH)
    return {
        "metadata_csv": metadata_csv,
        "metadata_source": metadata_csv,
        "inst_deploy_ID": inst_deploy_id,
        "instrument": instrument,
        "manual_qc_flags": list(globals().get("manual_qc_flags", [])),
    }


def require_ready_config(require_instrument=True):
    if require_instrument and not str(inst_deploy_id).strip():
        raise ValueError("Set inst_deploy_id to a valid AQD deployment identifier before running this cell.")
    return build_workflow_config()


def fmt_utc(value):
    timestamp = pd.to_datetime(value, utc=True, errors="coerce")
    if pd.isna(timestamp):
        return ""
    return timestamp.strftime("%Y-%m-%dT%H:%M:%SZ")

In [ ]:

# Optional `proc_1` time coverage overrides.
# Leave both as None to use the deployment metadata window.
proc1_time_start_override = None
proc1_time_end_override = None

# Optional manual QC windows for `proc_2`.
# Define these after reviewing `proc_1`; they are applied in `proc_2`.
manual_qc_flags = []

# Set the process umask so new output files are group-writable by default.
os.umask(0o002)


## ID lookup

This cell validates setup inputs and resolves metadata for the selected `inst_deploy_id` before any processing step.

In [ ]:
# Validate setup and print metadata for the selected deployment identifier.

database_lookup = importlib.reload(database_lookup)

inst_deploy_id = str(inst_deploy_id).strip()

instrument = str(instrument).strip().upper()

selected_metadata_row = None
selected_metadata_cfg = None
selected_metadata_lines = []
lookup_found = False
lookup_error = None

try:
    _, selected_metadata_row, selected_metadata_cfg, selected_metadata_lines = database_lookup.get_instrument_context(
        None,
        inst_deploy_id,
        deployment_id=None,
    )
    lookup_found = True
    print(f"Metadata for inst_deploy_id={inst_deploy_id}:")
    for line in selected_metadata_lines:
        print(line)
except ValueError as exc:
    lookup_error = str(exc)
    print(f"ID lookup failed for inst_deploy_id={inst_deploy_id}: {lookup_error}")
    print("Use `inst_deploy_id_finder.ipynb` to search candidate IDs before rerunning this cell.")


## Run `proc_1`

This stage does four main things in order:
1. resolves the raw source file and the trim window,
2. builds the proc_1 metadata and IMOS FV00 filename,
3. writes the proc_1 NetCDF to the proc_1 directory,
4. updates the metadata CSV with the saved proc_1 filename.

The filename is written back to the CSV only after the file has been created successfully.


In [ ]:
workflow_config = require_ready_config()
proc1_config = dict(workflow_config)

proc1_time_start_override = globals().get("proc1_time_start_override")
proc1_time_end_override = globals().get("proc1_time_end_override")

if proc1_time_start_override not in (None, ""):
    proc1_config["time_coverage_start"] = str(proc1_time_start_override)
if proc1_time_end_override not in (None, ""):
    proc1_config["time_coverage_end"] = str(proc1_time_end_override)

resolved_start = proc1_config.get("time_coverage_start", selected_metadata_row.get("time_coverage_start"))
resolved_end = proc1_config.get("time_coverage_end", selected_metadata_row.get("time_coverage_end"))
print(f"proc_1 trim window: {fmt_utc(resolved_start)} to {fmt_utc(resolved_end)}")

proc1_result = run_proc1(proc1_config, source_path=proc1_source_path)


## Proc_1 controls

Set the time window first. If you plan to review and flag data, define those settings here before running proc_1.



In [ ]:
workflow_config = require_ready_config()
if proc2_input is None:
    raise ValueError("Run the proc_1 save cell before proc_2.")

proc2_result = run_proc2(workflow_config, input_dataset=proc2_input)


## Run `imos_delivery`

This step is the final compliance gate for the IMOS workflow.
Only the IMOS FV01 `proc_2` product is treated as the final deliverable. The FV00 file remains an intermediate processed product and is not staged as a final delivery.


In [ ]:
workflow_config = require_ready_config()
delivery_result = run_imos_delivery(workflow_config)

# optional: make the final delivery directory shared/group-writable
proc2_output_path = Path(proc2_result["output_path"])
proc2_output_path.parent.chmod(0o2775)
subprocess.run(["chgrp", "1054842", str(proc2_output_path)], check=True)

print(f"FV01 path: {delivery_result['proc_2_delivery']}")
print(f"Delivery file: {delivery_result['imos_deliverables_file']}")
